# Donor churn classifier — Harbor of Hope / Lighthouse data

**Course:** IS 455 — Enterprise Machine Learning (CRISP-DM)


## 1. Business Understanding

**Modeling mode — PREDICTIVE.** We estimate whether a supporter will be **lapsed** (no gift in 180+ days as of the analysis reference date) so development can **prioritize stewardship** before revenue is lost. This is *not* causal proof of what *causes* churn; it is **discriminative** risk ranking for targeting.

**Business problem:** Harbor of Hope needs to flag at-risk donors early to schedule outreach and protect program funding.

**Success criteria:** ROC-AUC ≥ **0.65** on the held-out test set (reasonable for noisy nonprofit CRM data); we also review calibration for campaign planning.


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

df = pd.read_csv(DATA_DIR / 'donor_master.csv', low_memory=False)
print('shape', df.shape)
df.head()


In [ ]:
TARGET = 'is_churned'
num_cols = ['days_since_last_donation','total_lifetime_value','donation_frequency','num_campaigns','avg_gift_size','is_recurring_donor']
cat_cols = ['acquisition_channel','supporter_type']
print(df[num_cols].describe())
print('\nMissing:', df[num_cols+cat_cols+[TARGET]].isna().sum())


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
for i, c in enumerate(num_cols):
    df[c].hist(ax=axes[i], bins=20, color='#3498db')
    axes[i].set_title(c)
for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()
for c in cat_cols:
    plt.figure(figsize=(6,3))
    df[c].astype(str).value_counts().head(10).plot(kind='bar', color='#2ecc71')
    plt.title(c); plt.tight_layout(); plt.show()


In [ ]:
# Bivariate: numeric vs target correlation heatmap
nc = [TARGET] + [c for c in num_cols if c in df.columns]
sns.heatmap(df[nc].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation vs churn'); plt.tight_layout(); plt.show()
# Chi-square: categorical vs target
for c in cat_cols:
    ct = pd.crosstab(df[c].fillna('NA'), df[TARGET])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    print(c, 'chi2=', round(chi2,2), 'p=', round(p,4))
# ANOVA: numeric feature means across churn groups (Chapter 8-style group comparison)
for c in num_cols:
    a = df.loc[df[TARGET]==0, c].dropna()
    b = df.loc[df[TARGET]==1, c].dropna()
    if len(a)>2 and len(b)>2:
        print(c, 'ANOVA F', stats.f_oneway(a,b).statistic)


In [ ]:
# Feature engineering (business rules)
# num_campaigns already in master; log-transform heavy-tailed giving for stability
df = df.copy()
df['log_lifetime_value'] = np.log1p(df['total_lifetime_value'].clip(lower=0))
df['log_avg_gift'] = np.log1p(df['avg_gift_size'].clip(lower=0))
FEATURES = ['days_since_last_donation','log_lifetime_value','donation_frequency','num_campaigns',
            'acquisition_channel','supporter_type','is_recurring_donor','log_avg_gift']
model_df = df[FEATURES + [TARGET]].dropna(subset=[TARGET])
for c in ['days_since_last_donation','donation_frequency','num_campaigns','is_recurring_donor']:
    model_df[c] = model_df[c].fillna(model_df[c].median())
for c in ['log_lifetime_value','log_avg_gift']:
    model_df[c] = model_df[c].fillna(model_df[c].median())
for c in ['acquisition_channel','supporter_type']:
    model_df[c] = model_df[c].fillna('Unknown').astype(str)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
num_f = ['days_since_last_donation','log_lifetime_value','donation_frequency','num_campaigns','is_recurring_donor','log_avg_gift']
cat_f = ['acquisition_channel','supporter_type']
X = model_df[num_f + cat_f]
y = model_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
numeric_t = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_t = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                         ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric_t, num_f), ('cat', categorical_t, cat_f)])
print(X_train.shape, X_test.shape)


## 3. Modeling & Feature Selection


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
pipe_lr = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
pipe_rf = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))])
param_grid = {'clf__n_estimators': [100, 200], 'clf__max_depth': [4, 8, None], 'clf__min_samples_leaf': [1, 3]}
gs = GridSearchCV(pipe_rf, param_grid, scoring='roc_auc', cv=5, n_jobs=1, refit=True)
gs.fit(X_train, y_train)
pipe_lr.fit(X_train, y_train)
best_rf = gs.best_estimator_
print('Best RF params', gs.best_params_)


In [ ]:
# Feature importance: RF impurity; LR coefficients on preprocessed space (approximate)
rf_model = best_rf.named_steps['clf']
imp = rf_model.feature_importances_
feat_names = best_rf.named_steps['prep'].get_feature_names_out()
fi = pd.Series(imp, index=feat_names).sort_values(ascending=False).head(15)
print('Top RF importances:\n', fi)
lr_model = pipe_lr.named_steps['clf']
coef = pd.Series(np.ravel(lr_model.coef_), index=pipe_lr.named_steps['prep'].get_feature_names_out())
print('\nTop |coef| LR:\n', coef.reindex(coef.abs().sort_values(ascending=False).index).head(15))


## 4. Evaluation & Interpretation


In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
for name, mdl in [('Logistic', pipe_lr), ('RandomForest', best_rf)]:
    proba = mdl.predict_proba(X_test)[:,1]
    print(name, 'ROC-AUC', roc_auc_score(y_test, proba))
    print(classification_report(y_test, mdl.predict(X_test), digits=3))
fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_predictions(y_test, best_rf.predict_proba(X_test)[:,1], ax=ax, name='Tuned RF')
RocCurveDisplay.from_predictions(y_test, pipe_lr.predict_proba(X_test)[:,1], ax=ax, name='LR')
ax.plot([0,1],[0,1],'k--'); ax.set_title('ROC — test set only'); plt.tight_layout(); plt.show()
print('Confusion (RF):\n', confusion_matrix(y_test, best_rf.predict(X_test)))


In [ ]:
# Business readout
print('A higher ROC-AUC means the model ranks lapsed donors better — useful for sending a finite number of stewardship touches to the highest-risk supporters first.')


## 5. Causal / Relationship Analysis

**EXPLANATORY** companion model: **sklearn LogisticRegression** on dummy-coded predictors (train split only) — we report **exp(coefficient)** as a common **odds-ratio-style** readout for linear logits. This remains **associational**; correlation does **not** imply causation (unmeasured stewardship, life events, etc.).


In [ ]:
from sklearn.linear_model import LogisticRegression
X_ex = X_train.copy()
X_dm = pd.get_dummies(X_ex, drop_first=True).apply(pd.to_numeric, errors='coerce').fillna(0.0)
lr_ex = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42, solver='lbfgs')
lr_ex.fit(X_dm, y_train)
coef = pd.Series(lr_ex.coef_[0], index=X_dm.columns)
print('Intercept', lr_ex.intercept_[0])
print('\nexp(coef) (associational) top 12:')
print(np.exp(coef).sort_values(ascending=False).head(12))


## 6. Deployment Notes

**Artifact:** save the stronger **test-set** model (here: tuned Random Forest pipeline) with `joblib`.

**Product:** **At Risk** badge on the **donor profile** page in the admin app.


In [ ]:
import joblib
from sklearn.metrics import roc_auc_score
final = best_rf if roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1]) >= roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:,1]) else pipe_lr
out_path = MODEL_DIR / 'donor_churn_model.sav'
joblib.dump(final, out_path)
print('Saved', out_path)
loaded = joblib.load(out_path)
sample = X_test.iloc[:1]
print('P(churn):', loaded.predict_proba(sample)[0,1])
